In [99]:
import json
from pathlib import Path
from pprint import pprint
import numpy as np
import math  
import random 
from tqdm import tqdm

This notebook will solve a game called loldle (which is like wordle but for guessing league of legends champions based on attributes).

## Load Data

In [73]:
# Load the Loldle champion data JSON
json_path = Path('loldle-champ-data.json')
with json_path.open('r', encoding='utf-8') as f:
    champs = json.load(f)

N = len(champs)
N

163

In [6]:
# Add a derived attribute: release_year
for champ in champs:
    date_str = champ.get("release_date")
    if date_str:
        champ["release_year"] = int(date_str[:4])
    else:
        champ["release_year"] = None

# Quick check: first champ's year and all unique years
unique_years = sorted({c["release_year"] for c in champs if c["release_year"] is not None})
min(unique_years), max(unique_years)

(2009, 2022)

In [22]:
champ 

{'championName': 'Urf',
 'originalSpecies': ['Manatee'],
 'species': ['Manatee'],
 'regions': ['Runeterra'],
 'gender': 'Male',
 'positions': ['Top', 'Jungle', 'Middle', 'Support', 'Bottom'],
 'resource': 'Unknown',
 'release_date': '2010-04-01',
 'range_type': ['Melee'],
 'release_year': 2010}

Attributes: We use a total of 7 attributes that describe a champion.
* single_attrs: attributes with one possible categorical answer (for example, each champion uses exactly one resource). there are 2 responses to a guess: green (correct) or red (incorrect).
* ordered_attrs: attributes with one possible ordinal answer (for example, release year). there are 3 responses to a guess: green (correct) or earlier (the true year is earlier than guess) or later (the true year is after the guess year)
* list_attrs: attributes with many possible categorical answers (for example, a champion can be played in more than one position). there are 2 responses to a guess: green (exactly correct), yellow (partial match), or red (guess list has no overlap with correct list).

Multiplying the number of possible responses per attribute together, there are 972 total response outcomes.


In [16]:
single_attrs = ["resource", "gender"]
list_attrs = ["species", "positions", "regions", "range_type"]
ordered_attrs = ["release_year"]
all_attrs = single_attrs + list_attrs + ordered_attrs 
len(all_attrs)

7

In [24]:
# prepare cleaned dictionary 
clean_champ_dict = {}
for champ_spec in champs:
    name = champ_spec['championName']
    clean_champ_dict[name] = {}
    for attr in single_attrs:
        answer = champ_spec[attr]
        assert isinstance(answer, str)
        clean_champ_dict[name][attr] = answer 
    for attr in list_attrs:
        answer = champ_spec[attr]
        assert isinstance(answer, list)
        clean_champ_dict[name][attr] = answer
    for attr in ordered_attrs:
        answer = champ_spec[attr]
        assert isinstance(answer, int)
        clean_champ_dict[name][attr] = answer

In [69]:
with open("clean_champ_dict.json", "w") as f:
    json.dump(clean_champ_dict, f, indent=4)

In [26]:
attr2responses = {}
attr2type = {}
for attr in list_attrs:
    attr2responses[attr] = {"red", "green", "yellow"}
    attr2type[attr] = "list"
for attr in single_attrs:
    attr2responses[attr] = {"red", "green"}
    attr2type[attr] = "single"
for attr in ordered_attrs:
    attr2responses[attr] = {"green", "earlier", "later"}
    attr2type[attr] = "ordered"
pprint.pprint(attr2responses)
total_responses = int(np.prod([len(resp) for _, resp in attr2responses.items()]))
total_responses

{'gender': {'green', 'red'},
 'positions': {'yellow', 'green', 'red'},
 'range_type': {'yellow', 'green', 'red'},
 'regions': {'yellow', 'green', 'red'},
 'release_year': {'earlier', 'later', 'green'},
 'resource': {'green', 'red'},
 'species': {'yellow', 'green', 'red'}}


972

In [36]:
from itertools import product

keys = list(attr2responses.keys())
all_response_dicts = [
    dict(zip(keys, combo))
    for combo in product(*[attr2responses[k] for k in keys])
]
len(all_response_dicts)

972

Next, we need to filter the champion pool based on a guessed champion and the response. 

In [25]:
clean_champ_dict["Lux"]

{'resource': 'Mana',
 'gender': 'Female',
 'species': ['Human', 'Magicborn'],
 'positions': ['Middle', 'Support'],
 'regions': ['Demacia'],
 'range_type': ['Ranged'],
 'release_year': 2010}

In [ ]:
def compare_lists(x_list, y_list):
    if set(x_list) == set(y_list):
        return "equal"
    
    for x in x_list:
        if x in y_list:
            return "partial"
    
    return "disjoint"

In [43]:
def provide_feedback(guessed_champ, true_champ):
    response_dict = {}
    for attr, attr_type in attr2type.items():
        guessed_answer = clean_champ_dict[guessed_champ][attr]
        true_answer = clean_champ_dict[true_champ][attr]

        if attr_type == "single":
            if guessed_answer == true_answer:
                feedback = "green"
            else:
                feedback = "red"

        elif attr_type == "ordered":
            if guessed_answer == true_answer:
                feedback = "green"
            elif true_answer < guessed_answer:
                feedback = "earlier"
            elif true_answer > guessed_answer:
                feedback = "later"
        
        elif attr_type == "list":
            if compare_list(guessed_answer, true_answer) == "equal":
                feedback = "green"
            elif compare_list(guessed_answer, true_answer) == "partial":
                feedback = "yellow"
            else:
                feedback = "red"

        response_dict[attr] = feedback 
    return response_dict 


In [74]:
def filter_champs(guessed_champ, feedback):
    '''
    guessed_champ (str): name of guessed champion 
    response_dict (dict): dict mapping attribute to response 
    '''

    remaining_champs = []
    for candidate in clean_champ_dict.keys():
        hypothetical_feedback = provide_feedback(guessed_champ, candidate)
        if hypothetical_feedback == feedback:
            remaining_champs.append(candidate)

    return remaining_champs
        

In [71]:
fb = provide_feedback("Lux", "Vel'Koz")
fb

{'species': 'red',
 'positions': 'green',
 'regions': 'red',
 'range_type': 'green',
 'resource': 'green',
 'gender': 'red',
 'release_year': 'later'}

In [55]:
guessed_champ = "Lux"

In [64]:
response_dict = {
    'gender': 'red',
    'positions': 'yellow',
    'range_type': 'green',
    'regions': 'green',
    'release_year': 'green',
    'resource': 'green',
    'species': 'red'
}

In [72]:
remaining_champs = filter_champs(guessed_champ, fb)
remaining_champs

candidate: Lux
hypothetical
{'gender': 'green',
 'positions': 'green',
 'range_type': 'green',
 'regions': 'green',
 'release_year': 'green',
 'resource': 'green',
 'species': 'green'}
feedback
{'gender': 'red',
 'positions': 'green',
 'range_type': 'green',
 'regions': 'red',
 'release_year': 'later',
 'resource': 'green',
 'species': 'red'}
candidate: Aatrox
hypothetical
{'gender': 'red',
 'positions': 'red',
 'range_type': 'red',
 'regions': 'red',
 'release_year': 'later',
 'resource': 'red',
 'species': 'red'}
feedback
{'gender': 'red',
 'positions': 'green',
 'range_type': 'green',
 'regions': 'red',
 'release_year': 'later',
 'resource': 'green',
 'species': 'red'}
candidate: Ahri
hypothetical
{'gender': 'green',
 'positions': 'yellow',
 'range_type': 'green',
 'regions': 'red',
 'release_year': 'later',
 'resource': 'green',
 'species': 'red'}
feedback
{'gender': 'red',
 'positions': 'green',
 'range_type': 'green',
 'regions': 'red',
 'release_year': 'later',
 'resource': 'gre

["Vel'Koz", 'Xerath']

In [76]:
counts = []
for response_dict in all_response_dicts:
    remaining_champs = filter_champs(guessed_champ, response_dict)
    counts.append(len(remaining_champs))
assert sum(counts) == N  # every champ will get exactly one feedback. we are just splitting into buckets

In [87]:
def compute_entropy(counts):
    # entropy is - sum p log p
    assert sum(counts) == N

    total = 0 
    for n in counts:
        p = n / N 
        
        if p > 0:
            total += -1 * p * math.log2(p)
    
    return total 

In [88]:
compute_entropy(counts)

5.915618262730192

In [89]:
def compute_entropy_of_champion(guessed_champ):
    counts = []
    for response_dict in all_response_dicts:
        remaining_champs = filter_champs(guessed_champ, response_dict)
        counts.append(len(remaining_champs))
    # every champ will get exactly one feedback. we are just splitting into buckets
    assert sum(counts) == N  

    entropy = compute_entropy(counts)
    return entropy

In [ ]:
shuffled = random.sample(list(clean_champ_dict.keys()), 10)
shuffled 

['Kennen',
 'Samira',
 'Bard',
 'Ornn',
 'Akali',
 'Gwen',
 'Soraka',
 'Volibear',
 "Kog'Maw",
 'Orianna']

In [98]:
champ2entropy = {}
for champ in tqdm(clean_champ_dict):
    print(champ)
    entropy = compute_entropy_of_champion(champ)
    champ2entropy[champ] = entropy

  0%|          | 0/163 [00:00<?, ?it/s]

Lux


  1%|          | 1/163 [00:00<01:33,  1.74it/s]

Aatrox


  1%|          | 2/163 [00:01<01:31,  1.77it/s]

...

 99%|█████████▉| 161/163 [01:31<00:01,  1.77it/s]

K'Sante


 99%|█████████▉| 162/163 [01:31<00:00,  1.80it/s]

Urf


100%|██████████| 163/163 [01:32<00:00,  1.76it/s]


In [113]:
def display_dict(my_dict):
    for i, (name, value) in enumerate(sorted(my_dict.items(), key=lambda x: x[1], reverse=True)):
        print(f"({i:<3}) {name}: {value:.4f}")

In [114]:
clean_champ_dict["Talon"]

{'resource': 'Mana',
 'gender': 'Male',
 'species': ['Human'],
 'positions': ['Jungle', 'Middle'],
 'regions': ['Noxus'],
 'range_type': ['Melee'],
 'release_year': 2011}

In [116]:
clean_champ_dict["Bel\'Veth"]

{'resource': 'Manaless',
 'gender': 'Female',
 'species': ['Void-Being'],
 'positions': ['Jungle'],
 'regions': ['Void'],
 'range_type': ['Melee'],
 'release_year': 2022}

In [115]:
display_dict(champ2entropy)

(0  ) Talon: 6.1656
(1  ) Xin Zhao: 6.1523
(2  ) Brand: 6.1440
(3  ) Riven: 6.1140
(4  ) Cassiopeia: 6.0522
(5  ) Varus: 5.9965
(6  ) Vi: 5.9948
(7  ) Vayne: 5.9904
(8  ) Sona: 5.9870
(9  ) Darius: 5.9861
(10 ) Swain: 5.9725
(11 ) Jarvan IV: 5.9634
(12 ) Syndra: 5.9582
(13 ) Viktor: 5.9537
(14 ) Yorick: 5.9289
(15 ) Akali: 5.9276
(16 ) Graves: 5.9238
(17 ) Lux: 5.9156
(18 ) Fiora: 5.9071
(19 ) Karma: 5.9015
(20 ) Irelia: 5.8916
(21 ) LeBlanc: 5.8885
(22 ) Gragas: 5.8821
(23 ) Urgot: 5.8754
(24 ) Malzahar: 5.8617
(25 ) Ekko: 5.8257
(26 ) Zilean: 5.8214
(27 ) Taliyah: 5.7786
(28 ) Zed: 5.7770
(29 ) Annie: 5.7459
(30 ) Vladimir: 5.7342
(31 ) Ryze: 5.7307
(32 ) Quinn: 5.7184
(33 ) Lee Sin: 5.7163
(34 ) Garen: 5.7073
(35 ) Diana: 5.7054
(36 ) Draven: 5.6984
(37 ) Kayn: 5.6920
(38 ) Lissandra: 5.6904
(39 ) Ivern: 5.6786
(40 ) Shen: 5.6750
(41 ) Sejuani: 5.6750
(42 ) Caitlyn: 5.6716
(43 ) Leona: 5.6619
(44 ) Olaf: 5.6183
(45 ) Lucian: 5.6026
(46 ) Udyr: 5.5989
(47 ) Yasuo: 5.5983
(48 ) Kassad

In [117]:
18/162

0.1111111111111111